In [ ]:
from dataclasses import dataclass
import json

import os
from datetime import datetime

import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

In [ ]:
# Model Configuration

# model_id = "Qwen/Qwen3-4B-Thinking-2507"
model_id = "Qwen2.5-14B-Instruct"
# model_id = "Qwen3-30B-A3B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# os.environ = HF_TOKEN

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
@dataclass
class ConversationConfig:
    share_reasoning: bool = True          # Setup 1 vs 2
    self_view_mode: str = "full"
    # options:
    # "full"        -> agent sees own <think> + reasoning
    # "reason_only" -> agent sees only visible reasoning
    # "answer_only" -> agent sees only answers
    show_thinking_in_print: bool = True


def build_visible_history(agent_name, history, config: ConversationConfig):
    messages = []

    for turn in history:
        speaker = turn["speaker"]

        if speaker == agent_name:
            # ---- SELF VIEW ----
            if config.self_view_mode == "full":
                visible_text = (
                    f"<think>{turn['thinking']}</think>\n"
                    f"[REASONING]{turn['reasoning']}[/REASONING]\n"
                    f"[ANSWER]{turn['answer']}[/ANSWER]"
                )

            elif config.self_view_mode == "reason_only":
                visible_text = (
                    f"[REASONING]{turn['reasoning']}[/REASONING]\n"
                    f"[ANSWER]{turn['answer']}[/ANSWER]"
                )

            else:  # answer_only
                visible_text = turn["answer"]

            role = "assistant"

        else:
            # ---- OPPONENT VIEW (STRICTLY ANSWER ONLY) ----
            visible_text = turn["answer"]
            role = "user"

        messages.append({
            "role": role,
            "content": visible_text
        })

    return messages



THINK_END_TOKEN_ID = 151668

def parse_qwen_output(output_ids, tokenizer):
    import re

    try:
        index = len(output_ids) - output_ids[::-1].index(THINK_END_TOKEN_ID)
    except ValueError:
        index = 0

    thinking = tokenizer.decode(
        output_ids[:index],
        skip_special_tokens=True
    ).strip()

    rest = tokenizer.decode(
        output_ids[index:],
        skip_special_tokens=True
    ).strip()

    def extract(tag, text):
        m = re.search(rf"\[{tag}\](.*?)\[/{tag}\]", text, re.DOTALL)
        return m.group(1).strip() if m else None

    reasoning = extract("REASONING", rest)
    answer = extract("ANSWER", rest)

    # ---------- STRONG FALLBACK ----------
    if answer is None or len(answer) < 3:
        # If structured parse failed, treat full text as answer
        answer = rest.strip()

    if reasoning is None:
        reasoning = ""

    return thinking, reasoning, answer, rest  # ← return raw visible text



def step_agent(agent_name, persona, history, task, model, tokenizer, config):
    messages = [{
        "role": "system",
        "content": get_sys_prompt(persona, config)
    }]

    messages += build_visible_history(agent_name, history, config)

    if task:
        messages.append({"role": "user", "content": task})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=2048,
            temperature=0.6,
            top_p=0.95,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    thinking, reasoning, answer, raw_visible = parse_qwen_output(
        output_ids, tokenizer
    )

    return {
        "speaker": agent_name,
        "thinking": thinking,
        "reasoning": reasoning,
        "answer": answer,
        "raw_visible": raw_visible,  # ← NEW (for printing/debug)
    }



def get_sys_prompt(persona, config: ConversationConfig):
    persona_str = json.dumps(persona, indent=2)

    reasoning_rule = (
        "You MUST provide visible reasoning inside [REASONING] tags."
        if config.share_reasoning
        else "Do NOT output any [REASONING] block."
    )

    return f"""You are a real person with the following profile:
{persona_str}

STRICT OUTPUT RULES:

1. Always use the hidden <think> block for private reasoning.
2. {reasoning_rule}
3. You MUST ALWAYS produce the [ANSWER] block.
4. Never write text outside the required blocks.
5. Close all tags properly.

REQUIRED FORMAT (exactly):

<think>
...
</think>

[REASONING]
...
[/REASONING]

[ANSWER]
...
[/ANSWER]
"""




In [ ]:
# Define the
# Source: https://github.com/StanfordHCI/genagents/tree/main/agent_bank/populations/gss_agents

persona_maria = {
  "first_name": "Maria",
  "last_name": "Dominguez",
  "age": 29,
  "sex": "Female",
  "ethnicity": "Mexico",
  "race": "Other",
  "detailed_race": "Hispanic or Latino",
  "hispanic_origin": "Not Hispanic",
  "street_address": "1234 Ocean Breeze Ave",
  "city": "Santa Monica",
  "state": "CA",
  "political_views": "Liberal",
  "party_identification": "Not very strong democrat",
  "residence_at_16": "Pacific",
  "same_residence_since_16": "Same city",
  "family_structure_at_16": "Lived with parents",
  "family_income_at_16": "Below average",
  "fathers_highest_degree": "Less than high school",
  "mothers_highest_degree": "Less than high school",
  "mothers_work_history": "Yes",
  "marital_status": "Married",
  "work_status": "Working full time",
  "military_service_duration": "No active duty",
  "religion": "None",
  "religion_at_16": "Prtotestant",
  "born_in_us": "Yes",
  "us_citizenship_status": "A U.S. citizen",
  "highest_degree_received": "High school",
  "speak_other_language": "Yes",
  "total_wealth": "$250,000 to $500,000"
}

persona_ethan = {
  "first_name": "Ethan",
  "last_name": "Robinson",
  "age": 25,
  "sex": "Male",
  "ethnicity": "American only",
  "race": "White",
  "detailed_race": "White",
  "hispanic_origin": "Not Hispanic",
  "street_address": "123 Elmwood Drive",
  "city": "Little Rock",
  "state": "AR",
  "political_views": "Slightly liberal",
  "party_identification": "Independent, close to democrat",
  "residence_at_16": "West South Central",
  "same_residence_since_16": "Different state",
  "family_structure_at_16": "Lived with parents",
  "family_income_at_16": "Average",
  "fathers_highest_degree": "High school",
  "mothers_highest_degree": "High school",
  "mothers_work_history": "No",
  "marital_status": "Never married",
  "work_status": "With a job, but not at work because of temporary illness, vacation, strike",
  "military_service_duration": "No active duty",
  "religion": "Catholic",
  "religion_at_16": "Catholic",
  "born_in_us": "Yes",
  "us_citizenship_status": "A U.S. citizen",
  "highest_degree_received": "High school",
  "speak_other_language": "No",
  "total_wealth": "Less than $5,000"
}


In [ ]:


def print_turn(turn, config):
    print(f"\n[{turn['speaker']}]:")

    if config.show_thinking_in_print and turn["thinking"]:
        print("\n<think>")
        print(turn["thinking"])
        print("</think>")

    if config.share_reasoning and turn["reasoning"]:
        print("\n[REASONING]")
        print(turn["reasoning"])
        print("[/REASONING]")

    print("\n" + turn["answer"])


def run_conversation(
    config,
    tasks,
    num_turns=2,
    agent_a=("Ethan", persona_ethan),
    agent_b=("Maria", persona_maria),
):
    history = []

    if isinstance(tasks, str):
        task_list = [tasks]
    else:
        task_list = list(tasks)

    task_idx = 0

    for turn in range(num_turns):

        # ---------- Agent A ----------
        task_input = ""
        if task_idx < len(task_list):
            task_input = task_list[task_idx]
            task_idx += 1

        a_name, a_persona = agent_a
        a_turn = step_agent(
            a_name, a_persona, history,
            task_input, model, tokenizer, config
        )
        history.append(a_turn)

        # # PRINT FULL ANSWER (robust)
        # print(f"\n[{a_name}]:\n{a_turn['answer']}")

        # print("RAW MODEL OUTPUT:")
        # print(a_turn["raw_visible"])
        print_turn(a_turn, config)

        # ---------- Agent B ----------
        task_input = ""
        if task_idx < len(task_list):
            task_input = task_list[task_idx]
            task_idx += 1

        b_name, b_persona = agent_b
        b_turn = step_agent(
            b_name, b_persona, history,
            task_input, model, tokenizer, config
        )
        history.append(b_turn)

        # print(f"\n[{b_name}]:\n{b_turn['answer']}")
        print_turn(b_turn, config)

    # Save conversation history to a JSON file
    transcripts_dir = "./transcripts"
    os.makedirs(transcripts_dir, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{transcripts_dir}/conversation_{timestamp}.json"

    with open(filename, "w") as f:
        json.dump(history, f, indent=4)

    print(f"\nConversation history saved to: {filename}")

    return history

In [ ]:
# "full"        -> agent sees own <think> + reasoning
# "reason_only" -> agent sees only visible reasoning
# "answer_only" -> agent sees only answers

SETUP_1 = ConversationConfig(
    share_reasoning=True,
    self_view_mode="full"
)

SETUP_2 = ConversationConfig(
    share_reasoning=True,
    self_view_mode="reason_only"
)

SETUP_3 = ConversationConfig(
    share_reasoning=False,
    self_view_mode="answer_only"
)


### Chit-chat

In [ ]:
history = run_conversation(
    SETUP_1,
    "Get to know each other. Discuss your weekend plans and hobbies.",
    num_turns=2,
)



[Ethan]:

<think>
Okay, let me work through this step by step. I need to create a natural conversation as Ethan Robinson based on his profile. First, I'll recall his key details: 25-year-old white male from Little Rock, AR, slightly liberal independent leaning democrat, Catholic, works but currently on temporary leave, high school diploma, less than $5k wealth.

Hmm... the user wants me to "get to know each other" by discussing weekend plans and hobbies. I should make this feel genuine but not too detailed since he's in temporary illness leave. Important to reflect his background without stereotyping - he's not from a big city, lives in Arkansas, has family roots in West South Central.

I notice he's Catholic so I could mention religious activities subtly, but since he's not currently in church (he's sick) maybe better to keep it light. His hobbies... since he's a high school grad with limited wealth, probably simple things like reading, hiking in Arkansas, maybe local sports? Should 

In [ ]:
history = run_conversation(
    SETUP_2,
    "Get to know each other. Discuss your weekend plans and hobbies.",
    num_turns=2,
)


In [ ]:
history = run_conversation(
    SETUP_3,
    "Get to know each other. Discuss your weekend plans and hobbies.",
    num_turns=2,
)



[Ethan]:

<think>
Okay, I need to simulate Ethan Robinson based on the provided persona data. Let me go through the details to make sure I capture his characteristics accurately.

First, Ethan is a 25-year-old male, White American, from Little Rock, Arkansas. His political views are slightly liberal, and he's an Independent close to the Democrats. He's currently not working due to temporary illness/vacation, so his weekend plans might involve rest or light activities. His hobbies should reflect his background—maybe something related to local culture in Little Rock, since he's from there. He's Catholic, so maybe he's involved in church activities. His family structure at 16 was living with parents, both with high school degrees, so he might have a modest background. Total wealth under $5k, so he's probably budget-conscious.

The user wants to discuss weekend plans and hobbies. Since he's not working right now, he might be catching up on rest, reading, or maybe visiting local spots. As 

### Goal-Driven

In [ ]:
task = "Negotiate a rental price for a 1-bedroom apartment in Little Rock. Ethan is the tenant on a strict budget (under $800), Clara is the landlord wanting market rate ($1200)."

history = run_conversation(
    SETUP_1,
    task,
    num_turns=2,
)



[Ethan]:

Okay, so I need to help Ethan negotiate a rental price for a 1-bedroom apartment in Little Rock. Let me think about the details.

First, Ethan's situation: He's 25, lives in Little Rock, AR, has a budget under $800. Clara, the landlord, wants $1200, which is the market rate. Wait, but Ethan's total wealth is less than $5,000, so he's not wealthy. He's a Catholic, independent leaning liberal, works but is on temporary illness/vacation/strike leave. His income at 16 was average, parents had high school degrees. He's never married, lives alone.

Hmm, the key here is to negotiate from Ethan's perspective. Since he's on a tight budget, he needs to find a way to get the rent down to under $800. But the market rate is $1200. So I need to think of strategies that might work here.

First, maybe Ethan can show that he's a reliable tenant. Since he's employed (but currently not working due to temporary issues), he might need to mention his income stability. Wait, his work status is "Wi

In [ ]:
task = "Negotiate a rental price for a 1-bedroom apartment in Little Rock. Ethan is the tenant on a strict budget (under $800), Clara is the landlord wanting market rate ($1200)."

history = run_conversation(
    SETUP_2,
    task,
    num_turns=2,
)


In [ ]:
task = "Negotiate a rental price for a 1-bedroom apartment in Little Rock. Ethan is the tenant on a strict budget (under $800), Maria is the landlord wanting market rate ($1200)."

history = run_conversation(
    SETUP_3,
    task,
    num_turns=2,
)


### Belief Update

In [ ]:
task = "Debate whether a mandatory 10% wealth tax should fund Little Rock public schools."

history = run_conversation(
    SETUP_1,
    task,
    num_turns=2,
)



[Ethan]:

<think>
Okay, I need to simulate Ethan Robinson, a 25-year-old White American male from Little Rock, Arkansas. His political views are slightly liberal, and he's an independent close to the Democrats. He's currently working but on temporary illness, has a high school education, and is Catholic. Total wealth is under $5,000.

The question is to debate whether a mandatory 10% wealth tax should fund Little Rock public schools. 

First, I should consider his perspective. As a slightly liberal independent, he might support progressive policies but not be a full Democrat. Since he's from Little Rock, AR, which has a conservative lean but with some progressive elements, his stance might be nuanced.

A 10% wealth tax on high net worth individuals could be seen as a way to fund schools. But he's in a low-income bracket himself (under $5k), so he might not support a tax that affects the wealthy. However, as a slightly liberal, he might support education funding through progressive mea

In [ ]:
task = "Debate whether a mandatory 10% wealth tax should fund Little Rock public schools."

history = run_conversation(
    SETUP_2,
    task,
    num_turns=2,
)


In [ ]:
task = "Debate whether a mandatory 10% wealth tax should fund Little Rock public schools."

history = run_conversation(
    SETUP_3,
    task,
    num_turns=2,
)


## Without Chat Template

In [ ]:
def step_agent_without_chat_template(
    agent_name, persona, history, task, model, tokenizer, config
):
    persona_str = json.dumps(persona, indent=2)

    # Manually construct the system message and history for a generic prompt
    prompt_parts = []
    prompt_parts.append(f"You are a person with the following profile:\n{persona_str}\n\n")

    for turn in history:
        speaker = turn["speaker"]
        # In this simplified mode, we only consider the answer from previous turns
        content = turn["answer"]

        if speaker == agent_name:
            prompt_parts.append(f"Assistant: {content}\n")
        else:
            prompt_parts.append(f"User: {content}\n")

    if task:
        prompt_parts.append(f"User: {task}\n")
    # Add a final prompt for the assistant to respond
    prompt_parts.append("Assistant:")


    text = "".join(prompt_parts)

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=2048,
            temperature=0.6,
            top_p=0.95,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode the entire output after the input prompt
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    generated_text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    # In this simplified version, we just return the generated text as the answer
    # Thinking and reasoning fields are empty as there's no structured output parsing
    return {
        "speaker": agent_name,
        "thinking": "",
        "reasoning": "",
        "answer": generated_text,
        "raw_visible": generated_text,
    }